# Scenario 3: AI Agent with Tools — Monitoring with Galileo

---

## What You'll Learn

In this notebook, you will learn how to:

1. **Log agent sessions** with multiple conversation turns
2. **Log tool calls** (e.g., searching flights, booking hotels) as individual spans
3. **Nest spans** inside workflows to represent complex pipelines
4. **Use the `@log` decorator** to automatically instrument your Python functions
5. **Enable agentic metrics** to measure how well your agent performs

---

## What is an AI Agent?

An **AI Agent** is an LLM that can **take actions** — search the web, query databases, call APIs, send emails, etc. Unlike a simple chatbot that only generates text, an agent **makes decisions** about **which tools to use** and **when to use them**.

For example, a travel-planning agent might:
- Decide it needs to search for flights → calls a `search_flights` tool
- Decide it needs hotel options → calls a `search_hotels` tool
- Summarize the results using an LLM call
- Book the best option → calls a `book_flight` tool

Each of these steps generates data that Galileo can capture and analyze.

---

## Key Concepts

| Concept | What It Means |
|---|---|
| **Agent Span** | Logs the agent's decision-making process (e.g., "I need to search for flights") |
| **Tool Span** | Logs a specific tool invocation (e.g., calling `search_flights` or `book_hotel`) |
| **Workflow Span** | Groups related spans into a named pipeline (e.g., "ItineraryWorkflow" contains research + tool calls + summarization) |
| **`@log` Decorator** | Automatically creates spans from your Python functions — no manual logging code needed |
| **Agentic Metrics** | Scores that evaluate tool selection quality, action advancement, agent efficiency, and more |

---

> **Note:** This notebook uses **manual span logging** (no real LLM calls are made). This lets you see the trace structure without needing OpenAI API credits. The data is simulated so you can focus on learning the Galileo instrumentation patterns.

---

## Prerequisites

- Galileo API key set in your `.env` file
- The `galileo` Python package installed (handled by `uv sync`)
- Completion of Scenario 1 (Chatbot) is recommended but not required

### Step 0: Load Environment Variables

This cell loads your `.env` file, which contains your `GALILEO_API_KEY` (and optionally `OPENAI_API_KEY`). The code searches for the `.env` file in the current directory and the parent directory.

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


### Step 0b: Import Galileo Components

Here we import everything we need from the Galileo SDK:

- **`GalileoLogger`** — the main logger that lets you manually create traces and spans
- **`GalileoMetrics`** — an enum of all available metric types (used later to enable scoring)
- **`Message` / `MessageRole`** — data classes for representing chat messages (user, assistant, system)
- **`galileo_context`** — the global context manager that connects your code to a Galileo project
- **`galileo_log` (from `galileo.decorator`)** — this is the **key new import** for this notebook. It is a decorator that automatically wraps any Python function as a Galileo span, so you do not need to manually call `start_trace` / `add_span` / `conclude`.
- **`create_log_stream` / `get_log_stream` / `enable_metrics`** — utilities for managing log streams and enabling metric evaluation
- **`delete_project`** — cleanup utility

The constants (`PROJECT_NAME`, `LOG_STREAM_NAME`, `MODEL`) define where traces will be stored in the Galileo console.

## 1. Initialize Galileo

Same pattern as previous notebooks: connect to a Galileo project and log stream. This sets up the destination where all your agent traces will be stored. The console links let you view your traces in the Galileo web UI.

In [ ]:
from galileo import GalileoLogger, GalileoMetrics, Message, MessageRole, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.decorator import log as galileo_log
from galileo.log_streams import create_log_stream, enable_metrics, get_log_stream
from galileo.projects import delete_project

PROJECT_NAME = 'GalileoEval_S3_Agent'
LOG_STREAM_NAME = 'agent-monitoring'
MODEL = 'gpt-4o-mini'

## 2. Log a Multi-Turn Agent Session

This is the core of this notebook. We simulate a **travel planning agent** that has a 3-turn conversation with the user. Each turn is a separate trace, but they are all grouped under one **session** (`travel-agent-session-001`).

### How the 3 turns work:

| Turn | What Happens | Spans Logged |
|------|-------------|--------------|
| **Turn 1** | User asks "Plan me a 3-day trip to Tokyo" → Agent decides what to do | `agent_span` (agent reasoning) + `llm_span` (model response) |
| **Turn 2** | Agent uses tools to search for flights and hotels, then summarizes | `tool_span` (search_flights) + `tool_span` (search_hotels) + `llm_span` (summarization) |
| **Turn 3** | Agent books the flight and hotel | `tool_span` (book_flight) + `tool_span` (book_hotel) + `llm_span` (confirmation) |

### Span types used:

- **`add_agent_span()`** — Records the agent's decision-making (input: what it was asked, output: what it decided to do)
- **`add_tool_span()`** — Records a tool invocation (input: the function call, output: the tool's response)
- **`add_llm_span()`** — Records an LLM call (input: messages, output: assistant response)

### The session groups all 3 turns into one conversation:

```
Session: travel-agent-session-001
├── Trace 1: "Plan me a 3-day trip to Tokyo"
│   ├── AgentSpan: TravelPlannerAgent
│   └── LLMSpan: gpt-4o-mini
├── Trace 2: "Find flights from SFO to Tokyo"
│   ├── ToolSpan: search_flights
│   ├── ToolSpan: search_hotels
│   └── LLMSpan: gpt-4o-mini
└── Trace 3: "Book the ANA flight and hotel"
    ├── ToolSpan: book_flight
    ├── ToolSpan: book_hotel
    └── LLMSpan: gpt-4o-mini
```

Each `start_trace()` / `conclude()` pair creates one trace. The `start_session()` call at the top ties them all together so Galileo can evaluate the full conversation.

In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

log_stream = get_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
if not log_stream:
    log_stream = create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')

## 3. Log a Nested Workflow

A **Workflow** represents a named pipeline that contains sub-steps. Think of it as a container that groups related operations together.

In this example, we log: **Workflow → Agent → Tool → LLM** (a nested hierarchy).

```
Trace: "Generate a complete travel itinerary"
└── WorkflowSpan: ItineraryWorkflow
    └── AgentSpan: ResearchAgent
        └── ToolSpan: search_attractions
            └── LLMSpan: gpt-4o-mini (create day-by-day itinerary)
```

In the **Galileo Console**, you will see these as a **tree view** showing how sub-operations relate to parent operations. You can expand each node to see its input/output, duration, and any metrics.

**When to use workflows:** When your agent has a complex multi-step process. For example, "generate itinerary" involves research (agent reasoning), tool calls (searching attractions), and summarization (LLM call). Wrapping this in a workflow span makes it easy to see the entire pipeline as one logical unit.

In [5]:
logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)
logger.start_session(name='travel-agent-session-001')

logger.start_trace(input='Plan me a 3-day trip to Tokyo.')
logger.add_agent_span(
    input='Plan me a 3-day trip to Tokyo.',
    output="I'll search for flights, hotels, and attractions in Tokyo.",
    name='TravelPlannerAgent',
    duration_ns=100_000_000,
)
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Plan me a 3-day trip to Tokyo.')],
    output=Message(role=MessageRole.assistant, content="I'll help plan your Tokyo trip. Let me search for options."),
    model=MODEL,
)
logger.conclude(output="I'll help plan your Tokyo trip. Let me search for options.")

logger.start_trace(input='Find flights from SFO to Tokyo next month.')
logger.add_tool_span(
    input='search_flights(origin="SFO", destination="NRT", date="2026-04-10")',
    output='[{"airline":"ANA","price":850},{"airline":"JAL","price":920}]',
    duration_ns=300_000_000,
)
logger.add_tool_span(
    input='search_hotels(city="Tokyo", checkin="2026-04-10", nights=3)',
    output='[{"name":"Shinjuku Granbell","price_per_night":120},{"name":"Park Hyatt","price_per_night":450}]',
    duration_ns=250_000_000,
)
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Find flights and hotels for Tokyo trip.')],
    output=Message(role=MessageRole.assistant, content='ANA at $850 and Shinjuku Granbell at $120/night are the best value.'),
    model=MODEL,
)
logger.conclude(output='Found flights and hotels for Tokyo trip.')

logger.start_trace(input='Book the ANA flight and Shinjuku Granbell hotel.')
logger.add_tool_span(
    input='book_flight(airline="ANA", flight_id="NH107", passengers=1)',
    output='{"confirmation":"ANA-2026-XK9F","status":"confirmed"}',
    duration_ns=500_000_000,
)
logger.add_tool_span(
    input='book_hotel(hotel_id="shinjuku-granbell", checkin="2026-04-10", nights=3)',
    output='{"confirmation":"SG-88721","status":"confirmed"}',
    duration_ns=400_000_000,
)
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Book the ANA flight and Shinjuku Granbell.')],
    output=Message(role=MessageRole.assistant, content='Both booked. Flight ANA-2026-XK9F and hotel SG-88721 confirmed.'),
    model=MODEL,
)
logger.conclude(output='Trip booked successfully.')
logger.flush()
'3-turn agent session with tool calls'


'3-turn agent session with tool calls'

## 4. Log a Decorated Tool Flow

The **`@galileo_log` decorator** is the most ergonomic way to instrument your code. Instead of manually calling `start_trace()`, `add_span()`, and `conclude()`, you simply decorate your existing Python functions.

### How it works:

- **`@galileo_log(span_type='tool')`** — wraps the function as a Tool Span
- **`@galileo_log(span_type='agent')`** — wraps the function as an Agent Span
- **`@galileo_log(span_type='workflow')`** — wraps the function as a Workflow Span

The decorator automatically captures the function's **input arguments** and **return value** as the span's input/output.

### Nested calls create nested spans automatically:

When `plan_dinner()` calls `food_agent()`, which calls `search_restaurants()`, Galileo automatically nests the spans:

```
WorkflowSpan: plan_dinner
└── AgentSpan: food_agent
    └── ToolSpan: search_restaurants
```

**This is the recommended approach for production code.** Just decorate your existing functions and Galileo handles all the tracing plumbing for you.

In [6]:
logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)
logger.start_trace(input='Generate a complete travel itinerary.')
logger.add_workflow_span(
    input='Generate itinerary for Tokyo trip',
    output='3-day itinerary generated with attractions, restaurants, and transport.',
    name='ItineraryWorkflow',
    duration_ns=800_000_000,
)
logger.add_agent_span(
    input='Research top attractions in Tokyo',
    output='Found Senso-ji, Shibuya Crossing, Meiji Shrine, and Tsukiji Market.',
    name='ResearchAgent',
)
logger.add_tool_span(
    input='search_attractions(city="Tokyo", category="must-see")',
    output='["Senso-ji Temple","Shibuya Crossing","Meiji Shrine","Tsukiji Outer Market"]',
)
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Create a day-by-day itinerary.')],
    output=Message(role=MessageRole.assistant, content='Day 1: Asakusa. Day 2: Shibuya. Day 3: Meiji Shrine and Shinjuku.'),
    model=MODEL,
)
logger.conclude(output='Itinerary complete.')
logger.flush()
'Workflow trace logged'


'Workflow trace logged'

## 5. Enable Agentic Metrics

Now we turn on Galileo's built-in scoring for agent behavior. These metrics are automatically evaluated on your logged traces.

| Metric | What It Measures |
|--------|-----------------|
| **`action_advancement_luna`** | Does each agent step make **meaningful progress** toward the goal? A high score means the agent is not wasting steps or going in circles. |
| **`agent_efficiency`** | How efficiently does the agent solve the task? Fewer unnecessary steps = higher score. An agent that books a flight in 2 steps scores higher than one that takes 10. |
| **`agent_flow`** | Is the **sequence** of agent actions logical and coherent? For example, searching for flights before booking them is logical; booking before searching is not. |

After enabling these metrics, Galileo will score every new trace you log. You can view the scores in the Galileo Console alongside each trace.

In [7]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

@galileo_log(span_type='tool')
def search_restaurants(city: str, cuisine: str) -> str:
    return f'[{{"name":"Sukiyabashi Jiro","city":"{city}","cuisine":"{cuisine}"}}]'

@galileo_log(span_type='agent')
def food_agent(query: str) -> str:
    return f"Found restaurants: {search_restaurants(city='Tokyo', cuisine='sushi')}"

@galileo_log(span_type='workflow')
def plan_dinner(user_request: str) -> str:
    return f'Dinner plan: {food_agent(user_request)}'

result = plan_dinner('Find the best sushi in Tokyo')
galileo_context.flush()
result


'Dinner plan: Found restaurants: [{"name":"Sukiyabashi Jiro","city":"Tokyo","cuisine":"sushi"}]'

## 6. Enable Tool Metrics

These metrics focus specifically on how well the agent uses its tools.

| Metric | What It Measures |
|--------|-----------------|
| **`tool_error_rate`** | What fraction of tool calls result in errors? If the agent calls `search_flights` 10 times and 3 fail, the error rate is 0.3. Lower is better. |
| **`tool_selection_quality`** | Did the agent choose the **right tool** for the task? For example, if the user asks about flights and the agent calls `search_hotels` instead, this score will be low. |

> **Note:** Each call to `enable_metrics()` **replaces** the entire metric set for the log stream. So this call replaces the agentic metrics from Step 5 with these tool metrics. If you want both sets active, include all metrics in a single `enable_metrics()` call.

In [8]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.action_advancement_luna,
        GalileoMetrics.agent_efficiency,
        GalileoMetrics.agent_flow,
    ],
)


[]

## 7. Enable Reasoning and Session Metrics

These metrics evaluate the agent's reasoning quality and the overall conversation experience.

| Metric | What It Measures |
|--------|-----------------|
| **`reasoning_coherence`** | Is the agent's **chain of thought** logical? Does each reasoning step follow from the previous one? A high score means the agent is not contradicting itself or making illogical jumps. |
| **`action_completion_luna`** | Did the agent **successfully complete** the requested action? If the user asked to book a flight and the agent booked it, this scores high. If it only searched but never booked, it scores low. |
| **`conversation_quality`** | Overall quality of the multi-turn exchange. Considers factors like helpfulness, clarity, and whether the agent stayed on topic. |
| **`user_intent_change`** | Did the user's goal **change** during the conversation? This detects if the user shifted from "book a flight" to "cancel my subscription" mid-conversation, which may indicate confusion or poor agent guidance. |

These are especially useful for multi-turn sessions (like the 3-turn travel agent above) where you want to evaluate the entire conversation, not just individual steps.

In [9]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[GalileoMetrics.tool_error_rate, GalileoMetrics.tool_selection_quality],
)


[]

## 8. Clean Up the Demo Project


In [ ]:
delete_project(name=PROJECT_NAME)
